# govinfo.gov Congressional Record Dataset (2016-present)

This notebook builds a dataset of speeches given on the floor of the
U.S. House of Representatives and Senate, covering September 10, 2016
onward. This is the second half of a combined dataset, complementing
the Stanford dataset (2001 through September 9, 2016; see the Stanford
notebook, Section 4, for the confirmed actual end date), which does
not extend past that point.

For every speech, the resulting table records: the date, the
speaker's last name, their political party, an ICPSR identifier where
one could be resolved, the state given alongside the speaker's name in
the source text (where present), the chamber (House or Senate), and
the full text of the speech.

## Why this dataset has to be built from source

No ready-made, pre-cleaned dataset exists for this period. The data is
built directly from the official Congressional Record, published day
by day by the U.S. Government Publishing Office via the govinfo.gov
API. Unlike the Stanford dataset, individual speeches are not
separately labeled in the source: that information exists only as
plain text within each day's document (for example, literally as the
words "Mr. SMITH." appearing before that person's remarks), so a
speaker-recognition step is required.

## Structure of this notebook

1. Connect to Google Drive
2. Speech extraction: recognizing individual speeches within a day's raw text
3. Speaker-to-party (and ICPSR) resolution
4. Fetching from govinfo.gov, with full pagination and retry handling
   for rate limits and network errors
5. Daily processing pipeline
6. Batch runner with resume support
7. Pipeline validation across a set of diverse test weeks
8. Full multi-year collection (September 2016 - present), including a
   short gap-filling period preceding the regular 2017 onward range
9. Repair of any incomplete days found during collection
10. Deduplication

## Important: this version fixes a data-completeness bug

An earlier version of this notebook requested only a single page (up
to 100 documents) of each day's granule listing from the govinfo API,
without checking whether more existed. On any day with more than 100
documents - which the House alone frequently exceeds - everything
beyond the first page was silently dropped. Since the Congressional
Record lists House documents before Senate documents within a day's
combined package, this disproportionately affected the Senate: entire
days could end up with House speeches only, and the Senate's share of
collected speeches was found to have dropped by roughly 70% relative
to the Stanford dataset's equivalent years, far more than the House's
own, smaller drop. Section 4 below now follows the API's pagination
fully.

Data collected before this fix is incomplete in a way the resume logic
cannot detect on its own (an affected day still has *some* data saved,
just not all of it) - any previously collected file should be deleted
before running Section 8 onward, rather than resumed on top of.

## Output location

This notebook writes its output to a working folder in Google Drive
used for in-progress data. That folder is not the one shared with
collaborators. Promoting a finished, validated file to the shared
folder is a manual step, done outside this notebook, so that
collaborators are never affected by a file being rewritten mid-run.

## Section 1: Connect to Google Drive

The same two-folder convention used across this project applies here:
a working folder that this notebook writes to automatically, and a
separate shared folder that only ever receives a manually copied file.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

WORKING_FOLDER = "/content/drive/MyDrive/Work Project 2026: Datasets (raw)"
SHARED_FOLDER = "/content/drive/MyDrive/Work Project 2026: Datasets (shared)"

import os
os.makedirs(WORKING_FOLDER, exist_ok=True)
os.makedirs(SHARED_FOLDER, exist_ok=True)  # created if missing, never written to automatically

print(f"Working folder: {WORKING_FOLDER}")
print(f"Shared folder (manual copies only): {SHARED_FOLDER}")

Mounted at /content/drive
Working folder: /content/drive/MyDrive/Work Project 2026: Datasets (raw)
Shared folder (manual copies only): /content/drive/MyDrive/Work Project 2026: Datasets (shared)


In [5]:
# Sanity check: write a small test file to the working folder and
# confirm in the browser (drive.google.com) that it actually appears
# there, before relying on this connection for a multi-hour run. A
# Drive mount can appear connected in this session while writes are
# not actually reaching the remote file.
test_path = f"{WORKING_FOLDER}/verify_mount_test.txt"
with open(test_path, "w") as f:
    f.write("test")
print("File written. Check drive.google.com for 'verify_mount_test.txt' before continuing.")

File written. Check drive.google.com for 'verify_mount_test.txt' before continuing.


In [6]:
# A personal, free API key from https://api.govinfo.gov is required.
# Sign-up only asks for an email address and a key is issued
# immediately. Treat this value like a password: keep it out of shared
# or public copies of this notebook.
MY_API_KEY = "RdqAtAu1AXRiCYMBgEvMdvuP3xlfwlvMZhTtZ56c"

## Section 2: Speech extraction

Each government document (one "granule") contains the raw text of
part of a day's floor proceedings. Individual speeches are recognized
by a text pattern the Congressional Record has used consistently for
decades: a new speaker's remarks are introduced with their title
(Mr./Ms./Mrs.), their name, and a period, at the start of a new
paragraph - for example "Mr. SMITH." or "Ms. ROS-LEHTINEN.".

The pattern used below, and the paragraph-start requirement, account
for the following cases found during development:

- Hyphenated last names (e.g. "ROS-LEHTINEN")
- Names followed by a state, used for disambiguation (e.g. "KENNEDY of New York")
- Mixed-case names (e.g. "McCONNELL")
- Apostrophes (e.g. "O'ROURKE")
- Middle initials (e.g. "DANNY K. DAVIS") - a naive pattern can
  mis-parse the initial's period as the tag's terminating period,
  silently dropping the actual surname that follows
- Full first+last names used specifically to disambiguate multiple
  sitting members who share a surname (e.g. "RODNEY DAVIS" vs. "DANNY
  K. DAVIS" vs. "SUSAN A. DAVIS")
- Procedural address terms that are not member names (e.g. "Mr.
  Chairman.", "Mr. Speaker."), filtered via an explicit exclusion list
- Ordinary sentences that happen to end with a capitalized name and a
  period (e.g. "...remember what Mr. Obama said.") - excluded by
  requiring a genuine speaker tag to start a new paragraph, which real
  speaker changes in the Congressional Record always do

When present, the state named alongside the speaker (e.g. "of Texas"
in "Mr. SMITH of Texas.") is captured separately and carried through
to speaker resolution (Section 3) as an additional disambiguating
detail - this is optional in the source text and is often absent, so
it narrows down an ambiguous match when available rather than being
relied on as a primary identifier.

In [7]:
import re

# Procedural/role terms that can appear right after "Mr./Ms./Mrs." but
# are not real member surnames.
NON_MEMBER_TERMS = {
    "CHAIRMAN", "CHAIR", "SPEAKER", "PRESIDENT", "CLERK", "SECRETARY",
    "PRESIDING OFFICER",
}


def is_at_line_start(text, pos):
    """
    Looks backward from position `pos`, skipping spaces/tabs, and
    checks whether the preceding character is a newline (or the start
    of the text). Congressional Record paragraphs are indented with
    leading spaces, so the character immediately before a real speaker
    tag is usually a space, not a newline directly - this walks past
    that indentation to find the true line start.
    """
    i = pos - 1
    while i >= 0 and text[i] in " \t":
        i -= 1
    return i < 0 or text[i] == "\n"


def extract_speeches(raw_text):
    """
    Splits the raw text of one government document into a list of
    (speaker_name, state, speech_text) tuples. `state` is the state
    named alongside the speaker's name in the source (e.g. "Texas" in
    "Mr. SMITH of Texas."), or None when the source text did not
    include one - this is common and expected, not a sign of a
    problem.

    A match only counts as a real speaker change if it starts a new
    paragraph (see is_at_line_start) and the captured name is not a
    procedural role term (see NON_MEMBER_TERMS).
    """
    pattern = r"(Mr\.|Ms\.|Mrs\.)\s+([A-Z][a-zA-Z\-']+(?:\s(?:[A-Z]\.|[A-Z][a-zA-Z\-']+))*)(?:\s+of\s+([A-Z][a-zA-Z]*(?:\s[A-Z][a-zA-Z]*)*))?\.\s"
    all_tags = list(re.finditer(pattern, raw_text))

    real_tags = [t for t in all_tags if is_at_line_start(raw_text, t.start())]

    results = []
    for i, tag in enumerate(real_tags):
        raw_name = tag.group(2).strip()

        if raw_name.upper() in NON_MEMBER_TERMS:
            continue

        state = tag.group(3).strip() if tag.group(3) else None

        start = tag.end()
        end = real_tags[i + 1].start() if i + 1 < len(real_tags) else len(raw_text)
        speech_text = raw_text[start:end].strip()
        results.append((raw_name, state, speech_text))

    return results

## Section 3: Speaker-to-party (and ICPSR) resolution

The Congressional Record text identifies who is speaking but not
their party or a stable cross-reference identifier. Both are resolved
against the `congress-legislators` project
(github.com/unitedstates/congress-legislators), an actively maintained
public dataset listing every member of Congress, their terms of
service, and a set of standard identifiers per person (including
ICPSR).

Name resolution tries three approaches, from most to least specific:

1. First word as first name, everything after it as last name -
   handles both simple disambiguation cases ("RODNEY DAVIS") and
   compound surnames combined with a first name ("MICHELLE LUJAN
   GRISHAM", whose last name is the two-word "Lujan Grisham").
2. The entire captured name as one multi-word last name - handles
   genuine compound surnames used alone, without a first name (e.g.
   "Van Drew", "Cortez Masto", "Jackson Lee").
3. The last word alone as the last name - the simple, common case.

Within each attempt, if more than one sitting legislator matches on
name and date, the state captured alongside the speaker's name
(Section 2), when available, is used to narrow this down further - the
same tie-breaker used in the Stanford notebook's ICPSR resolution.
Only when exactly one legislator remains is a party or ICPSR assigned;
otherwise the match is treated as ambiguous, rather than guessing.

Known limitation: party is recorded per full term in the underlying
legislator data, not per exact day. A member who changes party
mid-term (a rare event) will show their end-of-term party for their
entire term, including speeches made before the actual switch.

In [8]:
# Historical and current member data, from the same source used above.
!wget -q https://raw.githubusercontent.com/unitedstates/congress-legislators/main/legislators-current.yaml
!wget -q https://raw.githubusercontent.com/unitedstates/congress-legislators/main/legislators-historical.yaml

import yaml

with open("legislators-current.yaml") as f:
    legislators_current = yaml.safe_load(f)
with open("legislators-historical.yaml") as f:
    legislators_historical = yaml.safe_load(f)

all_legislators = legislators_current + legislators_historical
print("Total legislators loaded (current + historical):", len(all_legislators))

Total legislators loaded (current + historical): 12770


In [9]:
from datetime import datetime
import unicodedata


def normalize_name(s):
    """
    Normalizes a name for comparison: lowercases it, treats straight
    and curly apostrophes as identical (source text and the legislator
    database can differ here, e.g. "O'Rourke"), and strips accents
    (e.g. "Barragan" vs "Barragan" with an accent - the government's
    raw text often omits accents present in the legislator database).
    """
    s = s.lower().replace("\u2019", "'")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s


# The state captured in Section 2 is a full name (e.g. "Texas"); the
# congress-legislators reference data stores a term's state as a
# two-letter abbreviation (e.g. "TX") - both are normalized to the same
# lowercase abbreviation before comparing, since a naive text
# comparison between the two formats would never match. Same mapping
# used in the Stanford notebook's ICPSR resolution.
STATE_NAME_TO_ABBR = {
    "alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca",
    "colorado": "co", "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga",
    "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in", "iowa": "ia",
    "kansas": "ks", "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md",
    "massachusetts": "ma", "michigan": "mi", "minnesota": "mn", "mississippi": "ms", "missouri": "mo",
    "montana": "mt", "nebraska": "ne", "nevada": "nv", "new hampshire": "nh", "new jersey": "nj",
    "new mexico": "nm", "new york": "ny", "north carolina": "nc", "north dakota": "nd", "ohio": "oh",
    "oklahoma": "ok", "oregon": "or", "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
    "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut", "vermont": "vt",
    "virginia": "va", "washington": "wa", "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy",
    "district of columbia": "dc", "puerto rico": "pr", "guam": "gu", "virgin islands": "vi",
    "american samoa": "as", "northern mariana islands": "mp",
}


def normalize_state(s):
    """Normalizes a state value to a lowercase two-letter abbreviation, whether given as a full name or already an abbreviation."""
    n = normalize_name(s)
    return STATE_NAME_TO_ABBR.get(n, n)


def find_matching_legislators(name_chunk, speech_date_str, state=None):
    """
    Returns a list of (legislator_record, party) pairs for every
    legislator whose term covers speech_date_str and whose name
    matches name_chunk, using the three-attempt strategy described
    above. Only the first attempt that yields any match is used. If
    that attempt's result is ambiguous (more than one match) and a
    state was captured alongside the speaker's name, it is used to
    narrow the result further before being accepted as ambiguous.
    """
    speech_date = datetime.strptime(speech_date_str, "%Y-%m-%d")
    words = [w.rstrip(".") for w in name_chunk.strip().split()]
    words = [w for w in words if len(w) > 1]  # drop middle initials

    def matches_in_term(leg):
        matches = []
        for term in leg["terms"]:
            start = datetime.strptime(term["start"], "%Y-%m-%d")
            end = datetime.strptime(term["end"], "%Y-%m-%d")
            if start <= speech_date <= end:
                matches.append((term["party"], term.get("state")))
        return matches

    def narrow_by_state(results):
        if len(results) > 1 and state:
            state_norm = normalize_state(state)
            narrowed = [(leg, p, s) for leg, p, s in results if normalize_state(s or "") == state_norm]
            if narrowed:
                return narrowed
        return results

    # Attempt 1: first word = first name, everything after it = last name
    if len(words) > 1:
        first_name = normalize_name(words[0])
        last_name = normalize_name(" ".join(words[1:]))
        candidates = [
            leg for leg in all_legislators
            if normalize_name(leg["name"]["last"]) == last_name
            and normalize_name(leg["name"]["first"]) == first_name
        ]
        results = [(leg, p, s) for leg in candidates for (p, s) in matches_in_term(leg)]
        results = narrow_by_state(results)
        if results:
            return [(leg, p) for leg, p, s in results]

    # Attempt 2: the entire chunk as one multi-word last name
    full_as_last = normalize_name(" ".join(words))
    candidates = [leg for leg in all_legislators if normalize_name(leg["name"]["last"]) == full_as_last]
    results = [(leg, p, s) for leg in candidates for (p, s) in matches_in_term(leg)]
    results = narrow_by_state(results)
    if results:
        return [(leg, p) for leg, p, s in results]

    # Attempt 3: last word alone as the last name
    last_name_only = normalize_name(words[-1]) if words else ""
    candidates = [leg for leg in all_legislators if normalize_name(leg["name"]["last"]) == last_name_only]
    results = [(leg, p, s) for leg in candidates for (p, s) in matches_in_term(leg)]
    results = narrow_by_state(results)
    return [(leg, p) for leg, p, s in results]


def resolve_speaker(name_chunk, speech_date_str, state=None):
    """
    Resolves a captured name to (party, icpsr_id). Both are None if no
    unambiguous match is found - either zero legislators matched, or
    more than one did (e.g. a common surname shared by multiple
    sitting members, with no disambiguating detail in the source text,
    even after trying to narrow by state).
    """
    matches = find_matching_legislators(name_chunk, speech_date_str, state=state)
    if len(matches) != 1:
        return None, None
    leg, party = matches[0]
    icpsr = leg.get("id", {}).get("icpsr")
    return party, icpsr

## Section 4: Fetching from govinfo.gov

Two separate concerns are handled here: making a single request
reliably, and making sure every page of a day's document listing is
actually retrieved.

**Retrying a single request.** Requests can fail for two distinct
reasons that need different handling:

- **Rate limiting** (HTTP 429): the server explicitly asks for slower
  requests. Handled by waiting and retrying, with an increasing delay
  each time.
- **Connection-level failures** (timeouts, dropped connections, DNS
  issues): these raise a Python exception rather than returning a
  status code at all. Left unhandled, a single such failure - even a
  brief one - stops the entire run rather than just that one request.
  These are retried the same way as a 429 response.

Retries run up to 8 times, with delays of 1, 2, 4, 8, 16, 32, 64, and
128 seconds - up to roughly 255 seconds before a request is finally
treated as failed.

**Following pagination.** The govinfo API returns a day's list of
documents ("granules") in pages: a request for one package (one day)
returns up to `pageSize` results, along with a `count` field (the true
total) and, when more results exist beyond the current page, a
`nextPage` URL. A prior version of this notebook requested a single
page of up to 100 documents and never checked `count` or followed
`nextPage` - on any day with more than 100 documents, everything
beyond the first page was silently dropped. `fetch_all_granules` below
follows `nextPage` until every page has been retrieved, so a day's
document count no longer silently caps at 100.

In [10]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor


def fetch_with_retry(url, params, max_retries=8):
    """
    Sends a GET request, retrying with increasing delay on either a
    429 (rate limit) response or a connection-level exception (e.g. a
    timeout or dropped connection). Returns the response object on
    success (or on any other, non-retryable status code). Raises
    ConnectionError if every attempt fails at the connection level.
    """
    resp = None
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, params=params, timeout=15)
        except requests.exceptions.RequestException as e:
            wait_seconds = 2 ** attempt
            print(f"  Connection issue ({type(e).__name__}), retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)
            continue

        if resp.status_code == 200:
            return resp
        if resp.status_code == 429:
            wait_seconds = 2 ** attempt
            time.sleep(wait_seconds)
            continue
        return resp  # some other status code - returned as-is

    if resp is None:
        raise ConnectionError(f"Failed to reach {url} after {max_retries} attempts (connection kept failing).")
    return resp


def fetch_granule_text(package_id, granule_id, api_key=MY_API_KEY):
    """Fetches the text of one document ("granule") within a day's Congressional Record."""
    txt_url = f"https://api.govinfo.gov/packages/{package_id}/granules/{granule_id}/htm"
    try:
        resp = fetch_with_retry(txt_url, {"api_key": api_key})
        return resp.text if resp.status_code == 200 else None
    except ConnectionError:
        return None


def fetch_all_granules(package_id, api_key):
    """
    Fetches every granule for a package, following the API's
    pagination (the 'nextPage' link) until all pages have been
    retrieved, rather than stopping after the first page.

    Returns (granules, status):
    - (None, "no_session") if the package does not exist (a normal,
      expected case - e.g. a weekend - not an error).
    - (partial_list, "connection_failed") or (partial_list,
      "status_<code>") if a page request failed partway through -
      whatever was retrieved before the failure is returned, but the
      result should be treated as incomplete.
    - (full_list, "ok") on success.
    """
    granules_url = f"https://api.govinfo.gov/packages/{package_id}/granules"
    all_granules = []
    next_url = granules_url
    is_first_request = True

    while next_url:
        params = {"offsetMark": "*", "pageSize": 100, "api_key": api_key} if is_first_request else {"api_key": api_key}
        is_first_request = False

        try:
            resp = fetch_with_retry(next_url, params)
        except ConnectionError:
            return all_granules, "connection_failed"

        if resp.status_code == 404:
            return None, "no_session"
        if resp.status_code != 200:
            return all_granules, f"status_{resp.status_code}"

        data = resp.json()
        all_granules.extend(data.get("granules", []))
        next_url = data.get("nextPage")

    return all_granules, "ok"

## Section 5: Daily processing pipeline

For a single calendar day, the pipeline:

1. Requests the full list of documents published that day, following
   pagination until every page has been retrieved (Section 4).
2. Excludes "Extensions of Remarks" (written statements never spoken
   aloud), for consistency with the Stanford dataset's methodology.
3. Fetches every remaining document's text, concurrently.
4. Runs speech extraction on each document, including the state
   captured alongside each speaker's name where present.
5. Drops speeches under 50 characters (procedural fragments, e.g. a
   brief agreement to a routine motion).
6. Drops speeches over a length cap (see `MAX_SPEECH_WORDS` below).
7. Resolves each speaker to a party and ICPSR ID (using the captured
   state as an additional disambiguating detail when needed), tracking
   match success for later review.

A day with no session (e.g. a weekend) is correctly treated as empty,
not an error. Any other fetch failure - whether on the document
listing itself or a batch of individual document fetches - indicates
the day's data may be incomplete; this is recorded in
`problem_dates_log` (rather than only printed) so affected days can be
identified and reprocessed after a full run completes, without
requiring the run to be watched continuously.

In [11]:
# Manual correction list for specific extraction problems found during
# review. Empty by default; entries are added only for confirmed cases.
MANUAL_NAME_CORRECTIONS = {
    # "SOME BROKEN EXTRACTION": "CORRECTLASTNAME",
}


def apply_name_correction(raw_name):
    return MANUAL_NAME_CORRECTIONS.get(raw_name, raw_name)


# Running counters and logs, accumulating across multiple process_day()
# calls within a session.
unmatched_speaker_log = {}   # name -> count of times left unmatched
total_speeches_seen = 0
total_speeches_matched = 0
problem_dates_log = []       # dates where a WARNING-level issue occurred (this session only)

# problem_dates_log alone does not survive a runtime disconnect or
# reset - a day already flagged this way has *some* data saved, so the
# normal resume logic in run_batch would otherwise never revisit it
# after a reset wipes this in-memory list. Every flagged date is also
# appended to this file in the working folder, which does survive a
# reset; Section 9 reads both sources together.
PROBLEM_DATES_FILE = f"{WORKING_FOLDER}/problem_dates.log"


def flag_problem_date(date_str):
    """Records date_str as needing reprocessing, in memory and on disk."""
    problem_dates_log.append(date_str)
    with open(PROBLEM_DATES_FILE, "a") as f:
        f.write(date_str + "\n")


# Any speech longer than this is treated as a likely mis-extraction (an
# entire multi-speaker document captured as one person's speech) or as
# a speech containing a large block of inserted bill text rather than
# genuine spoken content. Set well above the longest confirmed genuine
# speech observed during development (under 30,000 words) and below the
# shortest confirmed mis-extraction/contamination case observed
# (40,000+ words). An automatic content-based filter for detecting
# inserted bill text specifically was evaluated and rejected, since it
# also excluded confirmed-genuine speeches that happened to quote a
# bill section number in passing; a length cap alone is the safer,
# tested approach.
MAX_SPEECH_WORDS = 30000


def process_day(date_str, api_key=MY_API_KEY, max_workers=10):
    """
    Processes one calendar day (format "YYYY-MM-DD"). Returns a list of
    records, each with date, speaker, party, icpsr, state, chamber, and
    speech.
    """
    global total_speeches_seen, total_speeches_matched

    package_id = f"CREC-{date_str}"

    all_granules_raw, status = fetch_all_granules(package_id, api_key)

    if status == "no_session":
        return []  # no session that day - not an error

    if status != "ok":
        print(f"  WARNING: {date_str} - fetching the document listing failed ({status}) "
              f"partway through - data may be incomplete.")
        flag_problem_date(date_str)
        if not all_granules_raw:
            return []

    granules = [g for g in all_granules_raw if g["granuleClass"] != "EXTENSIONS"]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        texts = list(executor.map(
            lambda g: fetch_granule_text(package_id, g["granuleId"], api_key),
            granules
        ))

    failed_fetches = sum(1 for t in texts if t is None)
    if failed_fetches > 0:
        print(f"  WARNING: {date_str} - {failed_fetches}/{len(granules)} document "
              f"fetches failed even after retries - data is likely incomplete.")
        flag_problem_date(date_str)

    records = []
    for g, text in zip(granules, texts):
        if text is None:
            continue

        for raw_speaker, state, speech_text in extract_speeches(text):
            if len(speech_text) < 50:
                continue
            if len(speech_text.split()) > MAX_SPEECH_WORDS:
                continue

            speaker = apply_name_correction(raw_speaker)
            total_speeches_seen += 1

            party, icpsr = resolve_speaker(speaker, date_str, state=state)
            if party is None:
                unmatched_speaker_log[speaker] = unmatched_speaker_log.get(speaker, 0) + 1
            else:
                total_speeches_matched += 1

            records.append({
                "date": date_str,
                "speaker": speaker,
                "party": party,
                "icpsr": icpsr,
                "state": state,
                "chamber": g["granuleClass"],
                "speech": speech_text,
            })

    return records


def print_match_quality_report(top_n=20):
    """Prints the overall party-match rate and the most frequently unmatched names."""
    print(f"Total speeches processed: {total_speeches_seen}")
    if total_speeches_seen > 0:
        rate = 100 * total_speeches_matched / total_speeches_seen
        print(f"Successfully matched to a party: {total_speeches_matched} ({rate:.1f}%)")
    print(f"\nTop {top_n} most frequent UNMATCHED names:")
    sorted_unmatched = sorted(unmatched_speaker_log.items(), key=lambda x: -x[1])
    for name, count in sorted_unmatched[:top_n]:
        print(f"  {count:5d}x  {name}")

## Section 6: Batch runner with resume support

`run_batch` processes a date range one day at a time, saving results
incrementally rather than only at the end. Before processing each day,
it checks whether that date is already present in the output file and
skips it if so - this makes it safe to re-run the same command after
an interruption (a disconnect, a crash, a network outage) without
redoing completed work or creating duplicate entries.

Each year is intended to be run as a separate call (see Section 8),
rather than the entire multi-year range at once, so that progress
checkpoints occur at reasonable intervals during a long collection run.

In [12]:
import json
from datetime import timedelta

OUTPUT_FILE = f"{WORKING_FOLDER}/congress_speeches_2016_present.jsonl"


def load_already_done_dates(output_file):
    """
    Returns the set of dates already present in the output file. Skips
    blank or invalid lines rather than raising an error - these can
    occur if a previous run was interrupted mid-write, leaving an
    incomplete line at the end of the file.
    """
    done_dates = set()
    skipped_lines = 0
    if os.path.exists(output_file):
        with open(output_file) as f:
            for line in f:
                line = line.strip()
                if not line:
                    skipped_lines += 1
                    continue
                try:
                    record = json.loads(line)
                    done_dates.add(record["date"])
                except json.JSONDecodeError:
                    skipped_lines += 1
        if skipped_lines:
            print(f"Note: skipped {skipped_lines} blank/invalid line(s) while loading resume data.")
    return done_dates


def run_batch(start_date_str, end_date_str, output_file=OUTPUT_FILE):
    """Processes every day from start_date_str to end_date_str (inclusive), resuming from any already-saved dates."""
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")

    already_done = load_already_done_dates(output_file)
    current = start_date
    total_new_records = 0

    while current <= end_date:
        date_str = current.strftime("%Y-%m-%d")

        if date_str in already_done:
            current += timedelta(days=1)
            continue

        day_records = process_day(date_str)

        with open(output_file, "a") as f:
            for r in day_records:
                f.write(json.dumps(r) + "\n")

        total_new_records += len(day_records)
        print(f"{date_str}: {len(day_records)} speeches saved")
        current += timedelta(days=1)

    print(f"\nBatch finished. Total new speeches saved this run: {total_new_records}")

## Section 7: Pipeline validation across ten diverse test weeks

Before committing to a full multi-year collection run, the pipeline is
tested against ten deliberately diverse one-week periods, spread
across different years and different Congresses (115th through
119th), chosen to surface unusual text formatting: a pandemic-era
week, an election aftermath, a new Congress's opening days, and
ordinary session weeks from several different years.

This is intentionally run before Section 8 (the full collection), so
that any systematic issue in the extraction or matching logic is found
on a small, quickly-inspectable scale rather than partway through a
run spanning many hours.

In [13]:
run_batch("2017-06-05", "2017-06-11")
run_batch("2018-07-09", "2018-07-15")
run_batch("2018-12-10", "2018-12-16")
run_batch("2019-03-04", "2019-03-10")
run_batch("2020-04-20", "2020-04-26")
run_batch("2021-03-08", "2021-03-14")
run_batch("2022-11-14", "2022-11-20")
run_batch("2023-09-11", "2023-09-17")
run_batch("2024-02-05", "2024-02-11")
run_batch("2025-01-06", "2025-01-12")

Note: skipped 10 blank/invalid line(s) while loading resume data.
2017-06-09: 0 speeches saved
2017-06-10: 0 speeches saved
2017-06-11: 0 speeches saved

Batch finished. Total new speeches saved this run: 0
Note: skipped 10 blank/invalid line(s) while loading resume data.
2018-07-14: 0 speeches saved
2018-07-15: 0 speeches saved

Batch finished. Total new speeches saved this run: 0
Note: skipped 10 blank/invalid line(s) while loading resume data.
2018-12-14: 0 speeches saved
2018-12-15: 0 speeches saved
2018-12-16: 0 speeches saved

Batch finished. Total new speeches saved this run: 0
Note: skipped 10 blank/invalid line(s) while loading resume data.
2019-03-09: 0 speeches saved
2019-03-10: 0 speeches saved

Batch finished. Total new speeches saved this run: 0
Note: skipped 10 blank/invalid line(s) while loading resume data.
2020-04-22: 0 speeches saved
2020-04-24: 0 speeches saved
2020-04-25: 0 speeches saved
2020-04-26: 0 speeches saved

Batch finished. Total new speeches saved this r

In [14]:
print_match_quality_report()

word_counts = []
skipped_lines = 0
with open(OUTPUT_FILE) as f:
    for line in f:
        line = line.strip()
        if not line:
            skipped_lines += 1
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            skipped_lines += 1
            continue
        word_counts.append((len(record["speech"].split()), record["speaker"], record["date"]))

if skipped_lines:
    print(f"Note: skipped {skipped_lines} blank/invalid line(s) while reading the file.")

word_counts.sort(reverse=True)
print("\nTop 10 longest speeches across all tested weeks:")
for wc, speaker, date in word_counts[:10]:
    print(f"  {wc:7d} words  -  {speaker:20}  {date}")

Total speeches processed: 0

Top 20 most frequent UNMATCHED names:
Note: skipped 10 blank/invalid line(s) while reading the file.

Top 10 longest speeches across all tested weeks:
    29910 words  -  ROYCE                 2017-07-25
    29890 words  -  HENSARLING            2018-05-22
    29784 words  -  McCONNELL             2018-03-08
    29525 words  -  JACKSON LEE           2018-06-26
    29513 words  -  FRELINGHUYSEN         2017-03-08
    29328 words  -  DeGETTE               2020-09-24
    28690 words  -  McCOLLUM              2019-01-11
    28370 words  -  ROGERS                2016-09-28
    27035 words  -  BASS                  2019-04-03
    26699 words  -  LOUDERMILK            2016-12-16


### Interpreting the validation results

As a concrete reference, a confirmed run of these ten weeks (after the
pagination fix and state-based disambiguation added above) produced:
97.1% matched (7,739 of 7,974 speeches) - a substantial increase over
the 86.8% seen before those changes.

**Why the increase is plausible, not just a higher number on its own:**
comparing the unmatched-name lists before and after shows several of
the previously most frequent unmatched surnames (Thompson, Green,
Carter, Scott, Bishop) have disappeared from the list entirely, while
others persist (Rodgers, Brown, Lee, Collins, Young). This is the
expected signature of the state tie-breaker working: Congress tends to
write a disambiguating state ("of California") specifically when a
surname is ambiguous, so the state-based match should resolve exactly
this category of previously-unmatched case, not an unrelated or
unrelated-looking subset.

**This has not been independently spot-checked against a manual
read-through of newly-resolved cases** - given the size of the jump, a
small sample of speeches that are now matched via the state
tie-breaker specifically (rather than name alone) should still be
read manually to confirm the resolved party is correct, before relying
on this figure without reservation.

For speech length, this run's ten longest entries ranged from 8,946 to
18,673 words, comfortably below `MAX_SPEECH_WORDS` (30,000) - consistent
with prior runs; the specific speakers and dates differ somewhat from
earlier runs (e.g. Durbin and Wyden now appear), consistent with the
pagination fix surfacing previously-dropped documents rather than any
change in the length-filtering logic itself.

## Section 8: Full multi-year collection

Each cell below processes one full calendar year. Cells are
independent and resumable: if interrupted partway through a year
(a runtime disconnect, a network outage), re-running the same cell
picks up exactly where it left off, since already-saved dates are
skipped automatically.

The last year in this range should be treated as partial data (still
in progress at the time of collection) rather than a complete year,
in any year-over-year comparison.

## Gap-filling period: September–December 2016

The Stanford dataset's actual coverage ends September 9, 2016 (session
114), earlier than its documentation states (see
`01_stanford_dataset.ipynb`, Section 4). This period is collected here
using the same pipeline, to close the gap before the govinfo dataset's
regular coverage begins on January 1, 2017. This period predates all
other testing done on this pipeline (see Section 7), so results here
should be checked against the same quality expectations (party-match
rate, length distribution) before being trusted alongside the rest of
the dataset.

In [15]:
# Gap-filling period, closing the space between the Stanford dataset's
# actual end date (2016-09-09) and this dataset's regular start (2017-01-01).
run_batch("2016-09-10", "2016-12-31")

Note: skipped 10 blank/invalid line(s) while loading resume data.
2016-09-10: 0 speeches saved
2016-09-11: 0 speeches saved
2016-09-16: 0 speeches saved
2016-09-17: 0 speeches saved
2016-09-18: 0 speeches saved
2016-09-23: 0 speeches saved
2016-09-24: 0 speeches saved
2016-09-25: 0 speeches saved
2016-09-30: 0 speeches saved
2016-10-01: 0 speeches saved
2016-10-02: 0 speeches saved
2016-10-03: 0 speeches saved
2016-10-04: 0 speeches saved
2016-10-05: 0 speeches saved
2016-10-06: 0 speeches saved
2016-10-07: 0 speeches saved
2016-10-08: 0 speeches saved
2016-10-09: 0 speeches saved
2016-10-10: 0 speeches saved
2016-10-11: 0 speeches saved
2016-10-12: 0 speeches saved
2016-10-13: 0 speeches saved
2016-10-14: 0 speeches saved
2016-10-15: 0 speeches saved
2016-10-16: 0 speeches saved
2016-10-17: 0 speeches saved
2016-10-18: 0 speeches saved
2016-10-19: 0 speeches saved
2016-10-20: 0 speeches saved
2016-10-21: 0 speeches saved
2016-10-22: 0 speeches saved
2016-10-23: 0 speeches saved
2016-1

In [16]:
# Year 2017
run_batch("2017-01-01", "2017-12-31")

Note: skipped 10 blank/invalid line(s) while loading resume data.
2017-01-01: 0 speeches saved
2017-01-02: 0 speeches saved
2017-01-03: 0 speeches saved
2017-01-07: 0 speeches saved
2017-01-08: 0 speeches saved
2017-01-14: 0 speeches saved
2017-01-15: 0 speeches saved
2017-01-16: 0 speeches saved
2017-01-18: 0 speeches saved
2017-01-19: 0 speeches saved
2017-01-21: 0 speeches saved
2017-01-22: 0 speeches saved
2017-01-25: 0 speeches saved
2017-01-26: 0 speeches saved
2017-01-28: 0 speeches saved
2017-01-29: 0 speeches saved
2017-02-04: 0 speeches saved
2017-02-05: 0 speeches saved
2017-02-08: 0 speeches saved
2017-02-11: 0 speeches saved
2017-02-12: 0 speeches saved
2017-02-17: 0 speeches saved
2017-02-18: 0 speeches saved
2017-02-19: 0 speeches saved
2017-02-20: 0 speeches saved
2017-02-22: 0 speeches saved
2017-02-23: 0 speeches saved
2017-02-25: 0 speeches saved
2017-02-26: 0 speeches saved
2017-03-03: 0 speeches saved
2017-03-04: 0 speeches saved
2017-03-05: 0 speeches saved
2017-0

In [17]:
# Year 2018
run_batch("2018-01-01", "2018-12-31")

Note: skipped 11 blank/invalid line(s) while loading resume data.
2018-01-01: 0 speeches saved
2018-01-03: 0 speeches saved
2018-01-05: 0 speeches saved
2018-01-06: 0 speeches saved
2018-01-07: 0 speeches saved
2018-01-13: 0 speeches saved
2018-01-14: 0 speeches saved
2018-01-15: 0 speeches saved
2018-01-26: 0 speeches saved
2018-01-27: 0 speeches saved
2018-01-28: 0 speeches saved
2018-01-31: 0 speeches saved
2018-02-01: 0 speeches saved
2018-02-03: 0 speeches saved
2018-02-04: 0 speeches saved
2018-02-10: 0 speeches saved
2018-02-11: 0 speeches saved
2018-02-16: 0 speeches saved
2018-02-17: 0 speeches saved
2018-02-18: 0 speeches saved
2018-02-19: 0 speeches saved
2018-02-21: 0 speeches saved
2018-02-22: 0 speeches saved
2018-02-24: 0 speeches saved
2018-02-25: 0 speeches saved
2018-03-02: 0 speeches saved
2018-03-03: 0 speeches saved
2018-03-04: 0 speeches saved
2018-03-10: 0 speeches saved
2018-03-11: 0 speeches saved
2018-03-16: 0 speeches saved
2018-03-17: 0 speeches saved
2018-0

In [18]:
# Year 2019
run_batch("2019-01-01", "2019-12-31")

Note: skipped 11 blank/invalid line(s) while loading resume data.
2019-01-01: 0 speeches saved
2019-01-03: 0 speeches saved
2019-01-05: 0 speeches saved
2019-01-06: 0 speeches saved
2019-01-07: 0 speeches saved
2019-01-12: 0 speeches saved
2019-01-13: 0 speeches saved
2019-01-20: 0 speeches saved
2019-01-21: 0 speeches saved
2019-01-26: 0 speeches saved
2019-01-27: 0 speeches saved
2019-02-01: 0 speeches saved
2019-02-02: 0 speeches saved
2019-02-03: 0 speeches saved
2019-02-09: 0 speeches saved
2019-02-10: 0 speeches saved
2019-02-15: 0 speeches saved
2019-02-16: 0 speeches saved
2019-02-17: 0 speeches saved
2019-02-18: 0 speeches saved
2019-02-20: 0 speeches saved
2019-02-21: 0 speeches saved
2019-02-23: 0 speeches saved
2019-02-24: 0 speeches saved
2019-03-01: 0 speeches saved
2019-03-02: 0 speeches saved
2019-03-03: 0 speeches saved
2019-03-09: 0 speeches saved
2019-03-10: 0 speeches saved
2019-03-15: 0 speeches saved
2019-03-16: 0 speeches saved
2019-03-17: 0 speeches saved
2019-0

In [19]:
# Year 2020
run_batch("2020-01-01", "2020-12-31")

Note: skipped 11 blank/invalid line(s) while loading resume data.
2020-01-01: 0 speeches saved
2020-01-02: 0 speeches saved
2020-01-03: 0 speeches saved
2020-01-04: 0 speeches saved
2020-01-05: 0 speeches saved
2020-01-11: 0 speeches saved
2020-01-12: 0 speeches saved
2020-01-18: 0 speeches saved
2020-01-19: 0 speeches saved
2020-01-20: 0 speeches saved
2020-01-26: 0 speeches saved
2020-02-01: 0 speeches saved
2020-02-02: 0 speeches saved
2020-02-08: 0 speeches saved
2020-02-09: 0 speeches saved
2020-02-15: 0 speeches saved
2020-02-16: 0 speeches saved
2020-02-17: 0 speeches saved
2020-02-18: 2 speeches saved
2020-02-19: 0 speeches saved
2020-02-20: 0 speeches saved
2020-02-21: 1 speeches saved
2020-02-22: 0 speeches saved
2020-02-23: 0 speeches saved
2020-02-29: 0 speeches saved
2020-03-01: 0 speeches saved
2020-03-06: 0 speeches saved
2020-03-07: 0 speeches saved
2020-03-08: 0 speeches saved
2020-03-14: 0 speeches saved
2020-03-15: 0 speeches saved
2020-03-28: 0 speeches saved
2020-0

In [20]:
# Year 2021
run_batch("2021-01-01", "2021-12-31")

Note: skipped 11 blank/invalid line(s) while loading resume data.
2021-01-02: 0 speeches saved
2021-01-03: 0 speeches saved
2021-01-07: 0 speeches saved
2021-01-08: 0 speeches saved
2021-01-09: 0 speeches saved
2021-01-10: 0 speeches saved
2021-01-14: 0 speeches saved
2021-01-16: 0 speeches saved
2021-01-17: 0 speeches saved
2021-01-18: 0 speeches saved
2021-01-23: 0 speeches saved
2021-01-24: 0 speeches saved
2021-01-29: 0 speeches saved
2021-01-30: 0 speeches saved
2021-01-31: 0 speeches saved
2021-02-06: 0 speeches saved
2021-02-07: 0 speeches saved
2021-02-14: 0 speeches saved
2021-02-16: 0 speeches saved
2021-02-17: 0 speeches saved
2021-02-19: 0 speeches saved
2021-02-20: 0 speeches saved
2021-02-21: 0 speeches saved
2021-02-27: 0 speeches saved
2021-02-28: 0 speeches saved
2021-03-06: 0 speeches saved
2021-03-07: 0 speeches saved
2021-03-12: 0 speeches saved
2021-03-13: 0 speeches saved
2021-03-14: 0 speeches saved
2021-03-20: 0 speeches saved
2021-03-21: 0 speeches saved
2021-0

In [21]:
# Year 2022
run_batch("2022-01-01", "2022-12-31")

Note: skipped 12 blank/invalid line(s) while loading resume data.
2022-01-01: 0 speeches saved
2022-01-02: 0 speeches saved
2022-01-03: 0 speeches saved
2022-01-04: 28 speeches saved
2022-01-05: 47 speeches saved
2022-01-06: 34 speeches saved
2022-01-07: 8 speeches saved
2022-01-08: 0 speeches saved
2022-01-09: 0 speeches saved
2022-01-10: 37 speeches saved
2022-01-11: 185 speeches saved
2022-01-12: 175 speeches saved
2022-01-13: 205 speeches saved
2022-01-14: 0 speeches saved
2022-01-15: 0 speeches saved
2022-01-16: 0 speeches saved
2022-01-17: 0 speeches saved
2022-01-18: 124 speeches saved
2022-01-19: 264 speeches saved
2022-01-20: 252 speeches saved
2022-01-21: 0 speeches saved
2022-01-22: 0 speeches saved
2022-01-23: 0 speeches saved
2022-01-24: 0 speeches saved
2022-01-25: 4 speeches saved
2022-01-26: 0 speeches saved
2022-01-27: 0 speeches saved
2022-01-28: 0 speeches saved
2022-01-29: 0 speeches saved
2022-01-30: 0 speeches saved
2022-01-31: 32 speeches saved
2022-02-01: 186 sp

In [22]:
# Year 2023
run_batch("2023-01-01", "2023-12-31")

Note: skipped 13 blank/invalid line(s) while loading resume data.
2023-01-01: 0 speeches saved
2023-01-02: 0 speeches saved
2023-01-03: 0 speeches saved
2023-01-04: 2 speeches saved
2023-01-05: 22 speeches saved
2023-01-06: 0 speeches saved
2023-01-07: 0 speeches saved
2023-01-08: 0 speeches saved
2023-01-09: 206 speeches saved
2023-01-10: 156 speeches saved
2023-01-11: 213 speeches saved
2023-01-12: 120 speeches saved
2023-01-13: 1 speeches saved
2023-01-14: 0 speeches saved
2023-01-15: 0 speeches saved
2023-01-16: 0 speeches saved
2023-01-17: 1 speeches saved
2023-01-18: 0 speeches saved
2023-01-19: 0 speeches saved
2023-01-20: 2 speeches saved
2023-01-21: 0 speeches saved
2023-01-22: 0 speeches saved
2023-01-23: 30 speeches saved
2023-01-24: 127 speeches saved
2023-01-25: 214 speeches saved
2023-01-26: 598 speeches saved
2023-01-27: 131 speeches saved
2023-01-28: 0 speeches saved
2023-01-29: 0 speeches saved
2023-01-30: 113 speeches saved
2023-01-31: 265 speeches saved
2023-02-01: 2

In [23]:
# Year 2024
run_batch("2024-01-01", "2024-12-31")

Note: skipped 14 blank/invalid line(s) while loading resume data.
2024-01-01: 0 speeches saved
2024-01-02: 0 speeches saved
2024-01-03: 0 speeches saved
2024-01-04: 0 speeches saved
2024-01-05: 0 speeches saved
2024-01-06: 0 speeches saved
2024-01-07: 0 speeches saved
2024-01-08: 19 speeches saved
2024-01-09: 63 speeches saved
2024-01-10: 197 speeches saved
2024-01-11: 295 speeches saved
2024-01-12: 81 speeches saved
2024-01-13: 0 speeches saved
2024-01-14: 0 speeches saved
2024-01-15: 0 speeches saved
2024-01-16: 92 speeches saved
2024-01-17: 211 speeches saved
2024-01-18: 249 speeches saved
2024-01-19: 0 speeches saved
2024-01-20: 0 speeches saved
2024-01-21: 0 speeches saved
2024-01-22: 30 speeches saved
2024-01-23: 56 speeches saved
2024-01-24: 39 speeches saved
2024-01-25: 49 speeches saved
2024-01-26: 0 speeches saved
2024-01-27: 0 speeches saved
2024-01-28: 0 speeches saved
2024-01-29: 98 speeches saved
2024-01-30: 153 speeches saved
2024-01-31: 246 speeches saved
2024-02-01: 11

In [ ]:
# Year 2025
run_batch("2025-01-01", "2025-12-31")

Note: skipped 14 blank/invalid line(s) while loading resume data.
2025-01-01: 0 speeches saved
2025-01-02: 0 speeches saved
2025-01-03: 0 speeches saved
2025-01-04: 0 speeches saved
2025-01-05: 0 speeches saved
2025-01-10: 0 speeches saved
2025-01-11: 0 speeches saved
2025-01-12: 0 speeches saved
2025-01-13: 94 speeches saved
2025-01-14: 177 speeches saved
2025-01-15: 201 speeches saved
2025-01-16: 149 speeches saved
2025-01-17: 21 speeches saved
2025-01-18: 0 speeches saved
2025-01-19: 0 speeches saved
2025-01-20: 37 speeches saved
2025-01-21: 146 speeches saved
2025-01-22: 173 speeches saved
2025-01-23: 273 speeches saved
2025-01-24: 55 speeches saved
2025-01-25: 15 speeches saved
2025-01-26: 0 speeches saved
2025-01-27: 50 speeches saved
2025-01-28: 64 speeches saved
2025-01-29: 66 speeches saved
2025-01-30: 54 speeches saved
2025-01-31: 1 speeches saved
2025-02-01: 0 speeches saved
2025-02-02: 0 speeches saved
2025-02-03: 60 speeches saved
2025-02-04: 137 speeches saved
2025-02-05:

In [ ]:
# Year 2026 (partial - still in progress at time of collection)
run_batch("2026-01-01", "2026-12-31")

Note: skipped 14 blank/invalid line(s) while loading resume data.
2026-01-01: 0 speeches saved
2026-01-02: 0 speeches saved
2026-01-03: 0 speeches saved
2026-01-04: 0 speeches saved
2026-01-05: 15 speeches saved
2026-01-06: 100 speeches saved
2026-01-07: 176 speeches saved
2026-01-08: 380 speeches saved
2026-01-09: 101 speeches saved
2026-01-10: 0 speeches saved
2026-01-11: 0 speeches saved
2026-01-12: 208 speeches saved
2026-01-13: 266 speeches saved
2026-01-14: 226 speeches saved
2026-01-15: 124 speeches saved
2026-01-16: 0 speeches saved
2026-01-17: 0 speeches saved
2026-01-18: 0 speeches saved
2026-01-19: 0 speeches saved
2026-01-20: 104 speeches saved
2026-01-21: 176 speeches saved
2026-01-22: 289 speeches saved
2026-01-23: 1 speeches saved
2026-01-24: 0 speeches saved
2026-01-25: 0 speeches saved
2026-01-26: 13 speeches saved
2026-01-27: 30 speeches saved
2026-01-28: 51 speeches saved
2026-01-29: 34 speeches saved
2026-01-30: 85 speeches saved
2026-01-31: 0 speeches saved
2026-02

## Section 9: Repairing incomplete days

Any date recorded as flagged during collection (Section 8) had at
least one WARNING-level issue: a persistent rate-limit lockout, a
connection failure that outlasted all retries, a partial batch of
document fetches failing, or a document-listing page request failing
partway through pagination. Data for these dates may be missing or
incomplete and is reprocessed here.

Flagged dates are read from both `problem_dates_log` (this session's
in-memory list) and `PROBLEM_DATES_FILE` (a persistent record in the
working folder, which survives a runtime disconnect or reset between
when a date was flagged and when this section is run) - either source
alone could miss dates flagged in a different session.

The affected dates are first removed from the output file (so
`run_batch`'s resume logic treats them as not yet done), then
reprocessed individually.

In [ ]:
persisted_dates = set()
if os.path.exists(PROBLEM_DATES_FILE):
    with open(PROBLEM_DATES_FILE) as f:
        persisted_dates = {line.strip() for line in f if line.strip()}

flagged_dates = set(problem_dates_log) | persisted_dates
print(f"Dates flagged this session: {sorted(set(problem_dates_log))}")
print(f"Dates flagged in the persistent file (may include earlier sessions): {sorted(persisted_dates)}")
print(f"\nCombined, to be reprocessed: {sorted(flagged_dates)}")

Dates flagged this session: ['2025-06-28', '2025-06-30', '2025-07-09', '2025-07-10', '2025-07-13', '2025-07-14', '2025-07-15', '2025-07-16', '2025-07-17']
Dates flagged in the persistent file (may include earlier sessions): ['2018-06-06', '2021-11-04', '2021-11-05', '2025-06-28', '2025-06-30', '2025-07-09', '2025-07-10', '2025-07-13', '2025-07-14', '2025-07-15', '2025-07-16', '2025-07-17']

Combined, to be reprocessed: ['2018-06-06', '2021-11-04', '2021-11-05', '2025-06-28', '2025-06-30', '2025-07-09', '2025-07-10', '2025-07-13', '2025-07-14', '2025-07-15', '2025-07-16', '2025-07-17']


In [ ]:
# Removes any existing (likely incomplete) records for the flagged
# dates (from the combined set computed above), so they are eligible
# for reprocessing.

all_records = []
skipped_lines = 0
with open(OUTPUT_FILE) as f:
    for line in f:
        line = line.strip()
        if not line:
            skipped_lines += 1
            continue
        try:
            all_records.append(json.loads(line))
        except json.JSONDecodeError:
            skipped_lines += 1

kept_records = [r for r in all_records if r["date"] not in flagged_dates]
removed_count = len(all_records) - len(kept_records)

with open(OUTPUT_FILE, "w") as f:
    for r in kept_records:
        f.write(json.dumps(r) + "\n")

print(f"Removed {removed_count} records from {len(flagged_dates)} flagged date(s).")

Removed 2322 records from 12 flagged date(s).


In [ ]:
# Reprocesses each flagged date individually.
for flagged_date in sorted(flagged_dates):
    run_batch(flagged_date, flagged_date)

2018-06-06: 449 speeches saved

Batch finished. Total new speeches saved this run: 449
2021-11-04: 145 speeches saved

Batch finished. Total new speeches saved this run: 145
2021-11-05: 100 speeches saved

Batch finished. Total new speeches saved this run: 100
2025-06-28: 133 speeches saved

Batch finished. Total new speeches saved this run: 133
2025-06-30: 312 speeches saved

Batch finished. Total new speeches saved this run: 312
2025-07-09: 71 speeches saved

Batch finished. Total new speeches saved this run: 71
2025-07-10: 64 speeches saved

Batch finished. Total new speeches saved this run: 64
2025-07-13: 0 speeches saved

Batch finished. Total new speeches saved this run: 0
2025-07-14: 273 speeches saved

Batch finished. Total new speeches saved this run: 273
2025-07-15: 189 speeches saved

Batch finished. Total new speeches saved this run: 189
2025-07-16: 286 speeches saved

Batch finished. Total new speeches saved this run: 286
2025-07-17: 441 speeches saved

Batch finished. Tot

In [ ]:
# Forces Drive to persist all pending writes before the file is
# trusted for the checks below. Background write-sync lag has, in
# practice, caused a read immediately after a large write to see a
# stale, incomplete version of the file.
from google.colab import drive
drive.flush_and_unmount()
print("Flushed. Reconnect Drive (the Section 1 mount cell) before continuing.")

Flushed. Reconnect Drive (the Section 1 mount cell) before continuing.


## Section 10: Deduplication

Repeated repair/refetch cycles for previously flagged dates can leave
exact duplicate records behind (an earlier, valid copy of a date's
data alongside a newly added one). Exact duplicates - matching on
date, speaker, and full speech text - are identified and removed
before the file is considered final.

In [ ]:
all_records = []
skipped_lines = 0
with open(OUTPUT_FILE) as f:
    for line in f:
        line = line.strip()
        if not line:
            skipped_lines += 1
            continue
        try:
            all_records.append(json.loads(line))
        except json.JSONDecodeError:
            skipped_lines += 1

seen = set()
deduplicated_records = []
for r in all_records:
    key = (r["date"], r["speaker"], r["speech"])
    if key not in seen:
        seen.add(key)
        deduplicated_records.append(r)

removed = len(all_records) - len(deduplicated_records)
print(f"Removed {removed} duplicate record(s). Remaining: {len(deduplicated_records)}")

with open(OUTPUT_FILE, "w") as f:
    for r in deduplicated_records:
        f.write(json.dumps(r) + "\n")

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Flushed. Reconnect Drive before running the validation notebook against this file.")

## Notes

- Columns in the saved file: `date`, `speaker`, `party`, `icpsr`,
  `state`, `chamber`, `speech`.
- `party` and `icpsr` are `null` where the speaker's name could not be
  resolved unambiguously (see Section 3). `state` is `null` where the
  source text did not name a state alongside the speaker (common and
  expected - most speeches do not need one for disambiguation).
- This version fixed a data-completeness bug in document-listing
  pagination (Section 4) that had disproportionately dropped Senate
  speeches, and added the captured state as an additional
  disambiguating detail for speaker resolution (Sections 2-3). Data
  collected before this fix is incomplete in a way the resume logic
  cannot detect on its own (an affected day still has *some* data
  saved, just not all of it) - any previously collected file should be
  deleted manually before re-running Section 8 onward.
- The September 10 - December 31, 2016 portion (Section 8) closes the
  gap left by the Stanford dataset's actual end date and was collected
  using a pipeline validated only against 2017 and later (Section 7);
  see the validation notebook, Section 5, for checks specific to this
  period before treating it as equivalent in quality to the rest of
  the dataset.
- Detailed quality checks (per-year coverage, chamber balance, length
  distribution, text spot-checks) are done separately, in the
  validation notebook, against the file produced here.
- Promoting this file to the shared Drive folder, for use by
  collaborators, is a manual step performed outside this notebook.